# Cell 0: NB45 — Sağlam Bayes-Error Ceiling (Çoklu Yöntem + Floor-CI Disiplini)

## Amaç

NB40, PAH panelinde **k-NN tabanlı tek yöntemle** bir Bayes-error tavanı hesaplamış ve
`Bayes-F1(80/20)=0.6945`, `Model-F1=0.582`, `gap=+0.1125` bulmuştu. Bu, NB21–NB35'in "PAH'ta plato
gerçek, sinyal yok" bulgusuyla ve danışmanın floor-analiziyle (best=0.925 ≈ floor=0.905) **çelişiyor**.

`reports/literature_research_panel_improvements_2026-07-24.md` §3.7'nin işaret ettiği literatür
(Bayes Error Rate Estimation in Difficult Situations, arXiv 2506.03159) bu çelişkinin nedenini
gösteriyor: **k-NN tabanlı Bayes-error tahmini sınıf başına ~1000 örnek gerektiriyor** (düşük
boyutta bile); PAH'ın **62 benign** örneği ve yüzlerce feature'ı bu güven bandının çok altında.
Yani NB40'ın gap'i muhtemelen bir **k-NN artefaktı**, gerçek bir kazanç fırsatı değil.

Bu notebook, yol haritası deney #1'i uygular:

1. **Çoklu yöntem**: k-NN (çoklu k) + **MST-tabanlı GHP (Henze-Penrose / Friedman-Rafsky)** ile
   çapraz-doğrulama. GHP, k-NN'den bağımsız bir tahminci olduğu için k-NN'in kendi model
   performansıyla karışma riskini taşımaz.
2. **Çoklu feature-altküme**: Her panelde N=25 bootstrap feature-altkümesi (%75'i rastgele seçilir)
   üzerinden Bayes-error dağılımı çıkarılır — tahminin feature seçimine duyarlılığı doğrudan ölçülür.
3. **Floor-CI disiplini**: Floor F1 (`2*prev/(1+prev)`) da Bayes-F1 ile **aynı %80/20 bootstrap
   havuzunda, aynı N=50 tekrarla** hesaplanır ve aynı eksende karşılaştırılır — NB40'ın atladığı adım.
4. **Karar**: Eğer Bayes-F1 CI'ı floor-F1 CI'ıyla örtüşüyorsa (veya gap CI'ı 0'ı kapsıyorsa),
   PAH'taki "çelişki" bir artefakttır ve panel kapatılabilir. Örtüşmüyorsa gerçek bir marj var demektir.

**Not:** Bu notebook yalnızca **ölçüm sözleşmesini** düzeltir — yeni bir model eğitmez. NB39'daki
mevcut en iyi model F1'leri (BEST_MODEL_F1) sabit referans olarak kullanılır.


In [1]:
# Cell 1: Imports & Config

import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.spatial.distance import cdist
from scipy.stats import gaussian_kde

from fpdf import FPDF
from datetime import datetime

sys.path.insert(0, '/Users/tefe/teknofest_model/teknofest_model')
os.chdir('/Users/tefe/teknofest_model/teknofest_model')

from config import SEED, PROJECT_ROOT
from src.columns_real import (
    CAT_COLS, AA_COLS, TARGET_COL, ID_COL,
    get_constant_cols, get_duplicate_col_pairs,
    get_missing_mask_col_name, PANEL_INFO
)
from sklearn.metrics import f1_score, precision_score, recall_score, matthews_corrcoef

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v27_bayes_error_robust')
os.makedirs(RESULTS_DIR, exist_ok=True)

# NB39/NB32/NB21/NB20'den mevcut en iyi model F1'leri (final-realistic, %80/20 bootstrap)
BEST_MODEL_F1 = {
    'MASTER': 0.6379,
    'KANSER': 0.7300,
    'PAH': 0.5820,
    'CFTR': 0.8630
}

N_FEATURE_BOOT = 25       # feature-altkume bootstrap tekrari
FEATURE_FRAC = 0.75       # her tekrarda tutulan feature orani
N_DIST_BOOT = 50          # dagilim (80/20) bootstrap tekrari -- projenin standart protokolu
KS = (1, 3, 5, 7, 9, 15)  # cok-k taramasi (NB40'in 1,3,5,7'sinden genisletildi)


def fast_gower_matrix(X: np.ndarray, cat_features: np.ndarray) -> np.ndarray:
    '''
    Vektorize Gower mesafesi (numeric: onceden 0-1 normalize edilmis sutunlarin cdist-cityblock
    ortalamasi; kategorik: esitlik testi). `gower` paketi kategorik sutunlarda cok yavas bir
    Python-seviye implementasyon kullaniyor (MASTER n=2931 icin tek cagri ~124s olculdu --
    10 kategorik sutunla rastgele veride 5s'den 128s'ye cikiyor). Bu fonksiyon `gower.gower_matrix`
    ile SAYISAL OLARAK DOGRULANMISTIR (max fark ~3e-8, kayan nokta hassasiyeti seviyesinde);
    yalnizca performans icin yazilmistir, formul degismedi.
    '''
    X = np.asarray(X)
    n = X.shape[0]
    cat_features = np.asarray(cat_features)
    num_mask = ~cat_features

    total = np.zeros((n, n), dtype=np.float64)
    n_cols = X.shape[1]

    if num_mask.any():
        Xnum = X[:, num_mask].astype(np.float64)
        # build_gower_features zaten min-max [0,1] normalize ediyor -> cityblock (sum |a-b|) = gower numeric toplami
        total += cdist(Xnum, Xnum, metric='cityblock')

    if cat_features.any():
        Xcat = X[:, cat_features]
        for j in range(Xcat.shape[1]):
            col = Xcat[:, j]
            uniq, codes = np.unique(col, return_inverse=True)
            total += (codes[:, None] != codes[None, :]).astype(np.float64)

    return total / n_cols


print(f"SEED={SEED}")
print(f"Results dir: {RESULTS_DIR}")


SEED=42
Results dir: /Users/tefe/teknofest_model/teknofest_model/results/v27_bayes_error_robust


In [2]:
# Cell 2: load_panel(name) -- M3 missing strategy (NB40 ile ayni pipeline, degistirilmedi)

def load_panel(name: str) -> pd.DataFrame:
    '''
    Panelin bu notebook'taki hazirlik pipeline'i NB40 ile birebir aynidir --
    amac model degil OLCUM YONTEMI degistirmek. Ayni girdi uzerinde farkli
    (daha saglam) bir Bayes-error tahmincisi calistiriyoruz.
    '''
    fname = PANEL_INFO[name]['file']
    fpath = os.path.join(PROJECT_ROOT, 'data', 'real_data', fname)
    df = pd.read_csv(fpath)

    const_cols = get_constant_cols(df)
    if const_cols:
        df = df.drop(columns=const_cols)

    dup_pairs = get_duplicate_col_pairs(df)
    dup_cols_to_drop = {c2 for _, c2 in dup_pairs}
    if dup_cols_to_drop:
        df = df.drop(columns=list(dup_cols_to_drop))

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    missing_ratio = df[numeric_cols].isna().mean()
    high_missing_cols = missing_ratio[missing_ratio > 0.50].index.tolist()
    for col in high_missing_cols:
        df[get_missing_mask_col_name(col)] = df[col].isna().astype(int)
    for col in numeric_cols:
        if df[col].isna().sum() > 0:
            df[col].fillna(df[col].median(), inplace=True)

    n_pos = int((df[TARGET_COL] == 1).sum())
    n_neg = int((df[TARGET_COL] == 0).sum())
    print(f"{name}: shape={df.shape}, pos(patho)={n_pos}, neg(benign)={n_neg}, prev={n_pos/len(df):.4f}")
    return df


def build_gower_features(df: pd.DataFrame, feature_subset=None):
    '''Gower mesafesi icin feature hazirla; feature_subset verilirse yalnizca o kolonlari kullan.'''
    feature_cols = [c for c in df.columns if c not in [ID_COL, TARGET_COL]]
    if feature_subset is not None:
        feature_cols = [c for c in feature_cols if c in feature_subset]

    X = df[feature_cols].copy()
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    for col in numeric_cols:
        if not col.startswith('is_missing_'):
            mn, mx = X[col].min(), X[col].max()
            if mx > mn:
                X[col] = (X[col] - mn) / (mx - mn)

    cat_cols_present = [c for c in (CAT_COLS + AA_COLS) if c in X.columns]
    for col in cat_cols_present:
        X[col] = X[col].astype(str).replace('nan', 'MISSING')

    cat_features = np.array([c in cat_cols_present for c in X.columns])
    return X.values, cat_features, X.columns.tolist()


In [3]:
# Cell 3: knn_bayes_bounds -- coklu-k LOO + Cover-Hart bandi (vektorize: argpartition, tam argsort DEGIL)

def knn_bayes_bounds(X: np.ndarray, y: np.ndarray, ks=KS, cat_features=None):
    dist_matrix = fast_gower_matrix(X, cat_features)
    np.fill_diagonal(dist_matrix, np.inf)

    n = len(y)
    max_k = max(ks)
    # Her satirda sadece en yakin max_k komsuyu partition ile bul (O(n log k) / satir,
    # tam O(n log n) argsort'tan cok daha hizli), sonra bu alt-kumeyi mesafeye gore sirala.
    part_idx = np.argpartition(dist_matrix, max_k, axis=1)[:, :max_k]
    row_idx = np.arange(n)[:, None]
    part_dist = dist_matrix[row_idx, part_idx]
    order = np.argsort(part_dist, axis=1)
    sorted_neighbors = part_idx[row_idx, order]  # (n, max_k), mesafeye gore artan siralı

    knn_errors = {}
    for k in ks:
        neighbor_labels = y[sorted_neighbors[:, :k]]  # (n, k)
        preds = (neighbor_labels.mean(axis=1) > 0.5).astype(int)
        knn_errors[k] = float(np.mean(preds != y))

    R1 = knn_errors.get(1)
    if R1 is not None:
        R_star_high = R1
        if 1 - 2 * R1 < 0:
            R_star_low = 0.0
        else:
            R_star_low = (1 - np.sqrt(1 - 2 * R1)) / 2
    else:
        R_star_low = R_star_high = None

    return {"knn_errors": knn_errors, "R_star_low": R_star_low,
            "R_star_high": R_star_high, "dist_matrix": dist_matrix,
            "sorted_neighbors": sorted_neighbors, "max_k_cached": max_k}


## Cell 4: MST-Tabanlı GHP (Henze-Penrose / Friedman-Rafsky) Kesitmincisi

k-NN'den **bağımsız**, ağırlıklı-grafik tabanlı bir Bayes-error tahmincisi. Fikir: tüm örnekleri
(iki sınıf birlikte) tek bir tam-bağlı grafik olarak ele al, kenar ağırlıkları = Gower mesafesi;
bu grafiğin **minimum yayılan ağacını (MST)** kur. MST'deki kenarların ne kadarının **farklı
sınıflar arasında** olduğunu say — bu oran, sınıfların ne kadar iç içe geçtiğinin (dolayısıyla
Bayes-error'ın) bir ölçüsüdür (Friedman & Rafsky, 1979; Henze & Penrose, 1999).

Formül (Henze-Penrose divergence tahmini): `D_HP = 1 - 2*R_n / n` yaklaşık olarak sınıflar
arası ayrılabilirliği verir; buradan Bayes-error üst sınırı `R*_upper ≈ (1/2)*(1 - D_HP)` ile
türetilir (iki-örnek testi asimptotiklerinden). Tam kapalı-form yerine burada **ampirik oran**
kullanılıyor: `same_class_edge_ratio` yüksekse (sınıflar iyi ayrık) Bayes-error düşük, düşükse
(sınıflar karışık) Bayes-error yüksek.


In [4]:
# Cell 4: MST-tabanli GHP (vektorize edge-siniflandirma)

def mst_ghp_bayes_bound(dist_matrix: np.ndarray, y: np.ndarray) -> dict:
    '''
    Friedman-Rafsky / Henze-Penrose MST-tabanli Bayes-error tahmini.
    k-NN'den bagimsiz, cunku ayri-sinif kenar oranina dayanir (siniflandirma yapmiyor).
    '''
    n = len(y)
    mst = minimum_spanning_tree(dist_matrix).toarray()
    edges = np.argwhere(mst > 0)

    n_edges = len(edges)
    if n_edges == 0:
        return {"ghp_same_class_ratio": None, "ghp_bayes_upper": None, "n_edges": 0}

    diff_class_edges = int(np.sum(y[edges[:, 0]] != y[edges[:, 1]]))
    same_class_edges = n_edges - diff_class_edges
    same_class_ratio = same_class_edges / n_edges

    n1 = (y == 0).sum()
    n2 = (y == 1).sum()
    R_n = diff_class_edges
    ghp_divergence = 1 - (n * R_n) / (2 * n1 * n2) if (n1 > 0 and n2 > 0) else None

    if ghp_divergence is not None:
        ghp_divergence = np.clip(ghp_divergence, 0, 1)
        bayes_upper = 0.5 * (1 - ghp_divergence)
    else:
        bayes_upper = None

    return {
        "ghp_same_class_ratio": same_class_ratio,
        "ghp_divergence": ghp_divergence,
        "ghp_bayes_upper": bayes_upper,
        "n_edges": n_edges,
        "diff_class_edges": diff_class_edges,
    }


## Cell 5: Bootstrap Feature-Altküme Sağlamlığı

Her panelde Bayes-error tahminini **N=25 kez**, her seferinde feature'ların rastgele
**%75'ini** tutarak yeniden hesaplıyoruz (hem k-NN hem GHP için). Bu, NB40'ın "tek feature
kümesiyle tek sayı" yaklaşımının aksine, tahminin **feature seçimine ne kadar duyarlı**
olduğunu doğrudan gösterir. Geniş bir dağılım (yüksek std) = kırılgan tahmin = düşük güven.


In [5]:
# Cell 5: Feature-altkume bootstrap saglamligi (vektorize k=1 LOO + hizli Gower)

def feature_subset_robustness(df: pd.DataFrame, y: np.ndarray, n_boot=N_FEATURE_BOOT,
                               frac=FEATURE_FRAC, seed=SEED) -> dict:
    all_features = [c for c in df.columns if c not in [ID_COL, TARGET_COL]]
    n_keep = max(5, int(len(all_features) * frac))

    rng = np.random.RandomState(seed)
    knn1_errors = []
    ghp_uppers = []

    for b in range(n_boot):
        subset = rng.choice(all_features, size=n_keep, replace=False).tolist()
        X_sub, cat_feat_sub, _ = build_gower_features(df, feature_subset=subset)
        dist_sub = fast_gower_matrix(X_sub, cat_feat_sub)
        np.fill_diagonal(dist_sub, np.inf)

        # k=1 LOO -- vektorize: her satirin en yakin komsusu (argmin), tam Python dongusu yok
        nn_idx = np.argmin(dist_sub, axis=1)
        knn1_errors.append(float(np.mean(y[nn_idx] != y)))

        dist_sub_finite = dist_sub.copy()
        np.fill_diagonal(dist_sub_finite, 0)
        ghp_res = mst_ghp_bayes_bound(dist_sub_finite, y)
        if ghp_res["ghp_bayes_upper"] is not None:
            ghp_uppers.append(ghp_res["ghp_bayes_upper"])

    knn1_arr = np.array(knn1_errors)
    ghp_arr = np.array(ghp_uppers) if ghp_uppers else np.array([np.nan])

    return {
        "knn1_error_mean": knn1_arr.mean(), "knn1_error_std": knn1_arr.std(),
        "knn1_error_ci": np.percentile(knn1_arr, [2.5, 97.5]).tolist(),
        "ghp_upper_mean": np.nanmean(ghp_arr), "ghp_upper_std": np.nanstd(ghp_arr),
        "ghp_upper_ci": np.nanpercentile(ghp_arr, [2.5, 97.5]).tolist(),
        "n_boot": n_boot, "n_features_kept": n_keep, "n_features_total": len(all_features),
    }


## Cell 6: Bayes-Optimal F1 ve Floor-F1 — Aynı %80/20 Bootstrap Disiplininde

NB40'ın atladığı adım burada düzeltiliyor: **floor F1** (`2*prev/(1+prev)`, "hep pathogenic de"
baseline'ı) da Bayes-F1 ile **birebir aynı bootstrap döngüsünde**, aynı N=50 tekrarla, her
tekrarın **kendi prevalansından** hesaplanır. İkisi aynı eksende, aynı CI diliyle karşılaştırılır.


In [6]:
# Cell 6: Bayes-F1 ve Floor-F1 -- ayni 80/20 bootstrap havuzunda, ayni N ile (vektorize)

def bayes_and_floor_f1_8020(X, y, dist_matrix, sorted_neighbors=None, target_benign_frac=0.80,
                             n_boot=N_DIST_BOOT, k_for_pred=5, seed=SEED):
    n = len(y)
    rng = np.random.RandomState(seed)
    n_benign_total = (y == 0).sum()
    n_patho_total = (y == 1).sum()
    benign_ratio_natural = n_benign_total / n

    # k=5 icin en yakin komsu index'lerini ONCEDEN hesapla (tum n icin tek seferlik) --
    # her bootstrap tekrarinda ayni seyi yeniden hesaplamak yerine bir kez cikar, tekrarlarda indeksle.
    if sorted_neighbors is not None and sorted_neighbors.shape[1] >= k_for_pred:
        knn5_idx_all = sorted_neighbors[:, :k_for_pred]  # (n, k_for_pred), onceden hesaplanmis
    else:
        part_idx = np.argpartition(dist_matrix, k_for_pred, axis=1)[:, :k_for_pred]
        row_idx = np.arange(n)[:, None]
        part_dist = dist_matrix[row_idx, part_idx]
        order = np.argsort(part_dist, axis=1)
        knn5_idx_all = part_idx[row_idx, order]

    # Her ornek icin Bayes-optimal (k=5 coguluk oyu) tahmini TEK SEFERDE tum n icin hesapla
    y_pred_all = (y[knn5_idx_all].mean(axis=1) > 0.5).astype(int)

    bayes_f1, bayes_prec, bayes_rec, bayes_mcc = [], [], [], []
    floor_f1 = []

    benign_idx_all = np.where(y == 0)[0]
    patho_idx_all = np.where(y == 1)[0]

    for _ in range(n_boot):
        n_target_benign = int(len(benign_idx_all) * target_benign_frac / benign_ratio_natural)
        n_target_patho = int(len(patho_idx_all) * (1 - target_benign_frac) / (1 - benign_ratio_natural))
        n_target_benign = min(n_target_benign, len(benign_idx_all))
        n_target_patho = min(n_target_patho, len(patho_idx_all))

        sb = rng.choice(benign_idx_all, n_target_benign, replace=False)
        sp = rng.choice(patho_idx_all, n_target_patho, replace=False)
        boot_idx = np.concatenate([sb, sp])
        y_boot = y[boot_idx]
        y_pred = y_pred_all[boot_idx]  # onceden hesaplanmis tahminlerden indeksle -- yeniden argsort YOK

        bayes_f1.append(f1_score(y_boot, y_pred, zero_division=0))
        bayes_prec.append(precision_score(y_boot, y_pred, zero_division=0))
        bayes_rec.append(recall_score(y_boot, y_pred, zero_division=0))
        bayes_mcc.append(matthews_corrcoef(y_boot, y_pred))

        prev_this_boot = y_boot.mean()
        floor_f1.append(2 * prev_this_boot / (1 + prev_this_boot))

    bayes_f1_arr = np.array(bayes_f1)
    floor_f1_arr = np.array(floor_f1)
    gap_arr = bayes_f1_arr - floor_f1_arr

    return {
        "bayes_f1_mean": bayes_f1_arr.mean(), "bayes_f1_std": bayes_f1_arr.std(),
        "bayes_f1_ci": np.percentile(bayes_f1_arr, [2.5, 97.5]).tolist(),
        "bayes_precision_mean": np.mean(bayes_prec), "bayes_recall_mean": np.mean(bayes_rec),
        "bayes_mcc_mean": np.mean(bayes_mcc),
        "floor_f1_mean": floor_f1_arr.mean(), "floor_f1_std": floor_f1_arr.std(),
        "floor_f1_ci": np.percentile(floor_f1_arr, [2.5, 97.5]).tolist(),
        "bayes_minus_floor_gap_mean": gap_arr.mean(),
        "bayes_minus_floor_gap_ci": np.percentile(gap_arr, [2.5, 97.5]).tolist(),
    }


In [7]:
# Cell 7: run_panel(name) -- tum saglam analizleri birlestir

def run_panel_robust(name: str) -> dict:
    print(f"\n{'='*70}\nRunning (robust) panel: {name}\n{'='*70}")
    df = load_panel(name)
    y = df[TARGET_COL].values

    X, cat_features, feat_names = build_gower_features(df)

    print("k-NN coklu-k LOO + Cover-Hart...")
    knn_res = knn_bayes_bounds(X, y, ks=KS, cat_features=cat_features)

    print("MST-tabanli GHP (tum feature'lar)...")
    dist_full = knn_res["dist_matrix"].copy()
    np.fill_diagonal(dist_full, 0)
    ghp_res = mst_ghp_bayes_bound(dist_full, y)

    print(f"Feature-altkume bootstrap robustness (N={N_FEATURE_BOOT})...")
    robust_res = feature_subset_robustness(df, y, n_boot=N_FEATURE_BOOT, frac=FEATURE_FRAC)

    print(f"Bayes-F1 vs Floor-F1, ayni 80/20 bootstrap havuzunda (N={N_DIST_BOOT})...")
    dist_matrix_inf = knn_res["dist_matrix"]  # LOO icin diagonal=inf olan hali (self haric komsu)
    bf_res = bayes_and_floor_f1_8020(X, y, dist_matrix_inf, sorted_neighbors=knn_res["sorted_neighbors"],
                                      target_benign_frac=0.80, n_boot=N_DIST_BOOT)

    n_samples = len(y)
    n_benign = int((y == 0).sum())
    n_patho = int((y == 1).sum())

    model_f1 = BEST_MODEL_F1[name]
    gap_vs_model = bf_res["bayes_f1_mean"] - model_f1

    # Guvenilirlik bayragi: literatur k-NN icin sinif basi ~1000 ornek istiyor (dusuk boyutta bile)
    min_class_n = min(n_benign, n_patho)
    reliability = "LOW" if min_class_n < 100 else ("MEDIUM" if min_class_n < 500 else "HIGH")

    return {
        "panel": name, "n_samples": n_samples, "n_benign": n_benign, "n_patho": n_patho,
        "min_class_n": min_class_n, "reliability": reliability,
        "knn_errors": knn_res["knn_errors"],
        "R_star_low": knn_res["R_star_low"], "R_star_high": knn_res["R_star_high"],
        "ghp_same_class_ratio": ghp_res["ghp_same_class_ratio"],
        "ghp_divergence": ghp_res["ghp_divergence"],
        "ghp_bayes_upper": ghp_res["ghp_bayes_upper"],
        "robust_knn1_mean": robust_res["knn1_error_mean"],
        "robust_knn1_std": robust_res["knn1_error_std"],
        "robust_knn1_ci": robust_res["knn1_error_ci"],
        "robust_ghp_upper_mean": robust_res["ghp_upper_mean"],
        "robust_ghp_upper_std": robust_res["ghp_upper_std"],
        "robust_ghp_upper_ci": robust_res["ghp_upper_ci"],
        "bayes_f1_mean": bf_res["bayes_f1_mean"], "bayes_f1_ci": bf_res["bayes_f1_ci"],
        "bayes_precision_mean": bf_res["bayes_precision_mean"],
        "bayes_recall_mean": bf_res["bayes_recall_mean"],
        "bayes_mcc_mean": bf_res["bayes_mcc_mean"],
        "floor_f1_mean": bf_res["floor_f1_mean"], "floor_f1_ci": bf_res["floor_f1_ci"],
        "bayes_minus_floor_gap_mean": bf_res["bayes_minus_floor_gap_mean"],
        "bayes_minus_floor_gap_ci": bf_res["bayes_minus_floor_gap_ci"],
        "model_f1": model_f1, "gap_vs_model_mean": gap_vs_model,
    }


panel_results = {}
for panel_name in ['MASTER', 'KANSER', 'PAH', 'CFTR']:
    panel_results[panel_name] = run_panel_robust(panel_name)

print("\nTum paneller tamamlandi.")



Running (robust) panel: MASTER


MASTER: shape=(2931, 428), pos(patho)=2149, neg(benign)=782, prev=0.7332
k-NN coklu-k LOO + Cover-Hart...


MST-tabanli GHP (tum feature'lar)...


Feature-altkume bootstrap robustness (N=25)...


Bayes-F1 vs Floor-F1, ayni 80/20 bootstrap havuzunda (N=50)...

Running (robust) panel: KANSER


...

Bayes-F1 vs Floor-F1, ayni 80/20 bootstrap havuzunda (N=50)...

Running (robust) panel: PAH


PAH: shape=(372, 409), pos(patho)=310, neg(benign)=62, prev=0.8333
k-NN coklu-k LOO + Cover-Hart...
MST-tabanli GHP (tum feature'lar)...
Feature-altkume bootstrap robustness (N=25)...


Bayes-F1 vs Floor-F1, ayni 80/20 bootstrap havuzunda (N=50)...

Running (robust) panel: CFTR


CFTR: shape=(111, 312), pos(patho)=90, neg(benign)=21, prev=0.8108
k-NN coklu-k LOO + Cover-Hart...
MST-tabanli GHP (tum feature'lar)...
Feature-altkume bootstrap robustness (N=25)...


Bayes-F1 vs Floor-F1, ayni 80/20 bootstrap havuzunda (N=50)...

Tum paneller tamamlandi.


## Cell 8: Uzlaşma (Convergence) Kontrolü — k-NN vs GHP

İki bağımsız yöntemin (k-NN Cover-Hart, MST-GHP) Bayes-error tahminleri **birbirine yakınsa**,
tahmine güven artar. Uzaksa (veya feature-altküme CI'ları geniş çakışıyorsa), tahmin **kırılgandır**
ve NB40'ın tek-yöntem sonucuna güvenilmemelidir.


In [8]:
# Cell 8: Yontemler-arasi uzlasma tablosu

convergence_rows = []
for name, res in panel_results.items():
    knn1_r_high = res['R_star_high']  # k=1 hatasi (R* ust siniri)
    ghp_upper = res['ghp_bayes_upper']
    if knn1_r_high is not None and ghp_upper is not None:
        disagreement = abs(knn1_r_high - ghp_upper)
    else:
        disagreement = None

    convergence_rows.append({
        'Panel': name,
        'kNN-1 R*_high': f"{knn1_r_high:.4f}" if knn1_r_high is not None else "N/A",
        'GHP Bayes-upper': f"{ghp_upper:.4f}" if ghp_upper is not None else "N/A",
        'Disagreement': f"{disagreement:.4f}" if disagreement is not None else "N/A",
        'Feature-boot kNN1 CI': f"[{res['robust_knn1_ci'][0]:.3f}, {res['robust_knn1_ci'][1]:.3f}]",
        'Feature-boot GHP CI': f"[{res['robust_ghp_upper_ci'][0]:.3f}, {res['robust_ghp_upper_ci'][1]:.3f}]",
        'Reliability (n_min_class)': f"{res['reliability']} (n={res['min_class_n']})",
    })

convergence_df = pd.DataFrame(convergence_rows)
print("="*130)
print("YONTEMLER-ARASI UZLASMA (k-NN vs MST-GHP)")
print("="*130)
print(convergence_df.to_string(index=False))
convergence_df.to_csv(os.path.join(RESULTS_DIR, 'convergence_table.csv'), index=False)


YONTEMLER-ARASI UZLASMA (k-NN vs MST-GHP)
 Panel kNN-1 R*_high GHP Bayes-upper Disagreement Feature-boot kNN1 CI Feature-boot GHP CI Reliability (n_min_class)
MASTER        0.2845          0.3824       0.0979       [0.291, 0.309]      [0.365, 0.392]              HIGH (n=782)
KANSER        0.1985          0.2413       0.0428       [0.184, 0.226]      [0.225, 0.271]            MEDIUM (n=120)
   PAH        0.2124          0.3726       0.1602       [0.199, 0.228]      [0.345, 0.401]                LOW (n=62)
  CFTR        0.1892          0.3671       0.1779       [0.180, 0.261]      [0.311, 0.461]                LOW (n=21)


In [9]:
# Cell 9: Bayes-F1 vs Floor-F1 vs Model-F1 -- ayni CI disiplininde ozet tablo

summary_rows = []
for name, res in panel_results.items():
    bayes_ci = res['bayes_f1_ci']
    floor_ci = res['floor_f1_ci']
    gap_model = res['gap_vs_model_mean']
    gap_floor_ci = res['bayes_minus_floor_gap_ci']

    # CI'lar cakisiyor mu? (Bayes ile floor arasindaki fark istatistiksel olarak anlamli mi)
    ci_overlap = not (bayes_ci[0] > floor_ci[1] or floor_ci[0] > bayes_ci[1])
    gap_floor_includes_zero = gap_floor_ci[0] <= 0 <= gap_floor_ci[1]

    if gap_model < 0.03:
        model_decision = "DUR (tavanda)"
    elif gap_model > 0.10:
        model_decision = "DEVAM (marj var)"
    else:
        model_decision = "ARA (sinirli marj)"

    if ci_overlap or gap_floor_includes_zero:
        artefact_flag = "ARTEFAKT SUPHESI (Bayes~Floor, ayrisim yok)"
    else:
        artefact_flag = "Bayes > Floor (anlamli sinyal var)"

    summary_rows.append({
        'Panel': name,
        'Floor-F1 [CI]': f"{res['floor_f1_mean']:.3f} [{floor_ci[0]:.3f},{floor_ci[1]:.3f}]",
        'Bayes-F1 [CI]': f"{res['bayes_f1_mean']:.3f} [{bayes_ci[0]:.3f},{bayes_ci[1]:.3f}]",
        'Model-F1': f"{res['model_f1']:.3f}",
        'Gap(Bayes-Model)': f"{gap_model:.3f}",
        'Gap(Bayes-Floor) [CI]': f"{res['bayes_minus_floor_gap_mean']:.3f} [{gap_floor_ci[0]:.3f},{gap_floor_ci[1]:.3f}]",
        'Model-Karari': model_decision,
        'Bayes-Tavan-Guvenilirligi': artefact_flag,
        'Reliability': res['reliability'],
    })

summary_df = pd.DataFrame(summary_rows)
print("="*160)
print("NIHAI OZET: FLOOR vs BAYES vs MODEL (ayni %80/20 bootstrap CI disiplininde)")
print("="*160)
print(summary_df.to_string(index=False))
summary_df.to_csv(os.path.join(RESULTS_DIR, 'final_summary_robust.csv'), index=False)
print(f"\nKaydedildi: {os.path.join(RESULTS_DIR, 'final_summary_robust.csv')}")


NIHAI OZET: FLOOR vs BAYES vs MODEL (ayni %80/20 bootstrap CI disiplininde)
 Panel       Floor-F1 [CI]       Bayes-F1 [CI] Model-F1 Gap(Bayes-Model) Gap(Bayes-Floor) [CI]     Model-Karari                   Bayes-Tavan-Guvenilirligi Reliability
MASTER 0.600 [0.600,0.600] 0.656 [0.648,0.668]    0.638            0.018   0.057 [0.048,0.068]    DUR (tavanda)          Bayes > Floor (anlamli sinyal var)        HIGH
KANSER 0.562 [0.562,0.562] 0.726 [0.698,0.758]    0.730           -0.004   0.164 [0.136,0.196]    DUR (tavanda)          Bayes > Floor (anlamli sinyal var)      MEDIUM
   PAH 0.705 [0.705,0.705] 0.701 [0.685,0.715]    0.582            0.119 -0.004 [-0.020,0.010] DEVAM (marj var) ARTEFAKT SUPHESI (Bayes~Floor, ayrisim yok)         LOW
  CFTR 0.677 [0.677,0.677] 0.755 [0.727,0.772]    0.863           -0.108   0.078 [0.050,0.095]    DUR (tavanda)          Bayes > Floor (anlamli sinyal var)         LOW

Kaydedildi: /Users/tefe/teknofest_model/teknofest_model/results/v27_bayes_error_rob

## Cell 10: PAH Özel Yorumu

PAH'ın merkezi sorusunu doğrudan yanıtlıyoruz: NB40'ın gap=+0.1125'i bir k-NN artefaktı mı,
yoksa gerçek bir kazanç fırsatı mı?


In [10]:
# Cell 10: PAH ozel karar

pah_res = panel_results['PAH']
print("="*70)
print("PAH ÇELIŞKI ÇÖZÜMÜ")
print("="*70)
print(f"n_benign={pah_res['n_benign']} (literatur guvenilirlik esigi ~100-1000 altinda -> reliability={pah_res['reliability']})")
print(f"kNN-1 R*_high: {pah_res['R_star_high']:.4f}")
print(f"GHP Bayes-upper: {pah_res['ghp_bayes_upper']:.4f}" if pah_res['ghp_bayes_upper'] is not None else "GHP: N/A")
print(f"Feature-boot kNN1 CI: {pah_res['robust_knn1_ci']}")
print(f"Feature-boot GHP CI: {pah_res['robust_ghp_upper_ci']}")
print(f"\nFloor-F1 (80/20 bootstrap): {pah_res['floor_f1_mean']:.4f} {pah_res['floor_f1_ci']}")
print(f"Bayes-F1 (80/20 bootstrap): {pah_res['bayes_f1_mean']:.4f} {pah_res['bayes_f1_ci']}")
print(f"Model-F1 (NB21 best): {pah_res['model_f1']:.4f}")
print(f"Gap(Bayes-Model): {pah_res['gap_vs_model_mean']:.4f}  <-- NB40'ta +0.1125 bulunmustu")

bayes_ci = pah_res['bayes_f1_ci']
floor_ci = pah_res['floor_f1_ci']
ci_overlap = not (bayes_ci[0] > floor_ci[1] or floor_ci[0] > bayes_ci[1])

print(f"\nBayes-F1 CI ile Floor-F1 CI cakisiyor mu? {ci_overlap}")
if ci_overlap or pah_res['reliability'] == 'LOW':
    print(">>> SONUC: PAH'taki NB40 gap'i, floor-CI disiplini uygulaninca artefakt olarak")
    print("    degerlendirilmelidir (dusuk guvenilirlik / floor ile orusen CI). PAH plato bulgusu")
    print("    (NB21-NB35, danisman floor-analizi) GECERLIGINI KORUR.")
else:
    print(">>> SONUC: Bayes-F1 CI'i floor'dan anlamli sekilde ayrisiyor -- gercek bir marj olabilir,")
    print("    reverse-distribution + TabPFN denemeden 'PAH kapandi' denemez (yol haritasi #2, #3).")


PAH ÇELIŞKI ÇÖZÜMÜ
n_benign=62 (literatur guvenilirlik esigi ~100-1000 altinda -> reliability=LOW)
kNN-1 R*_high: 0.2124
GHP Bayes-upper: 0.3726
Feature-boot kNN1 CI: [0.19946236559139785, 0.22849462365591397]
Feature-boot GHP CI: [0.3454838709677419, 0.40064516129032257]

Floor-F1 (80/20 bootstrap): 0.7048 [0.7047619047619048, 0.7047619047619048]
Bayes-F1 (80/20 bootstrap): 0.7012 [0.6846278593376579, 0.714975845410628]
Model-F1 (NB21 best): 0.5820
Gap(Bayes-Model): 0.1192  <-- NB40'ta +0.1125 bulunmustu

Bayes-F1 CI ile Floor-F1 CI cakisiyor mu? True
>>> SONUC: PAH'taki NB40 gap'i, floor-CI disiplini uygulaninca artefakt olarak
    degerlendirilmelidir (dusuk guvenilirlik / floor ile orusen CI). PAH plato bulgusu
    (NB21-NB35, danisman floor-analizi) GECERLIGINI KORUR.


In [11]:
# Cell 11: Gorsellestirmeler

fig_paths = []
panel_names = list(panel_results.keys())

def err_bars(means, cis):
    '''yerr'i her zaman negatif olmayan olarak hesapla (float hassasiyeti nedeniyle
    mean CI sinirina esit/cok yakin oldugunda -1e-16 gibi degerler cikabilir).'''
    lo = [max(0.0, m - ci[0]) for m, ci in zip(means, cis)]
    hi = [max(0.0, ci[1] - m) for m, ci in zip(means, cis)]
    return [lo, hi]

# Figure 1: Floor-F1 vs Bayes-F1 vs Model-F1, hepsi CI ile (ayni CI disiplini)
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(panel_names))
width = 0.25

floor_means = [panel_results[p]['floor_f1_mean'] for p in panel_names]
floor_cis = [panel_results[p]['floor_f1_ci'] for p in panel_names]
floor_errs = err_bars(floor_means, floor_cis)

bayes_means = [panel_results[p]['bayes_f1_mean'] for p in panel_names]
bayes_cis = [panel_results[p]['bayes_f1_ci'] for p in panel_names]
bayes_errs = err_bars(bayes_means, bayes_cis)

model_means = [panel_results[p]['model_f1'] for p in panel_names]

ax.bar(x - width, floor_means, width, yerr=floor_errs, label='Floor-F1 (80/20 boot, CI)',
       color='gray', alpha=0.7, capsize=4)
ax.bar(x, bayes_means, width, yerr=bayes_errs, label='Bayes-F1 (80/20 boot, CI)',
       color='darkorange', alpha=0.8, capsize=4)
ax.bar(x + width, model_means, width, label='Model-F1 (NB39/32/21/20)', color='steelblue', alpha=0.8)

ax.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
ax.set_title('Floor vs Bayes-Ceiling vs Model — Aynı %80/20 Bootstrap CI Disiplini', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(panel_names, fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'fig1_floor_bayes_model.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
fig_paths.append(fig_path)
plt.close()

# Figure 2: Yontemler-arasi uzlasma (kNN-1 vs GHP), feature-boot CI'lariyla
fig, ax = plt.subplots(figsize=(11, 6))
knn_means = [panel_results[p]['robust_knn1_mean'] for p in panel_names]
knn_cis = [panel_results[p]['robust_knn1_ci'] for p in panel_names]
knn_errs = err_bars(knn_means, knn_cis)
ghp_means = [panel_results[p]['robust_ghp_upper_mean'] for p in panel_names]
ghp_cis = [panel_results[p]['robust_ghp_upper_ci'] for p in panel_names]
ghp_errs = err_bars(ghp_means, ghp_cis)

ax.bar(x - width/2, knn_means, width, yerr=knn_errs, label='k-NN-1 error (feature-boot CI)',
       color='seagreen', alpha=0.8, capsize=4)
ax.bar(x + width/2, ghp_means, width, yerr=ghp_errs, label='GHP Bayes-upper (feature-boot CI)',
       color='indianred', alpha=0.8, capsize=4)
ax.set_ylabel('Bayes-Error Tahmini', fontsize=12, fontweight='bold')
ax.set_title('Yöntemler-Arası Uzlaşma: k-NN vs MST-GHP (Feature-Altküme Bootstrap CI)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(panel_names, fontsize=11)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'fig2_method_convergence.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
fig_paths.append(fig_path)
plt.close()

# Figure 3: k-NN error curve coklu-k (6 k degeri, NB40'in 4'unden genis)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for idx, name in enumerate(panel_names):
    res = panel_results[name]
    ax = axes[idx]
    ks_sorted = sorted(res['knn_errors'].keys())
    errors = [res['knn_errors'][k] for k in ks_sorted]
    ax.plot(ks_sorted, errors, marker='o', linewidth=2, markersize=7, color='steelblue')
    ax.set_xlabel('k', fontsize=10, fontweight='bold')
    ax.set_ylabel('LOO Error', fontsize=10, fontweight='bold')
    ax.set_title(f"{name} (n={res['n_samples']}, min_class={res['min_class_n']})", fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.set_ylim([0, 1])
plt.suptitle('k-NN LOO Hata Egrisi (k=1..15)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'fig3_knn_curve_extended.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
fig_paths.append(fig_path)
plt.close()

# Figure 4: Gap(Bayes-Floor) CI -- 0'i kapsiyor mu (artefakt testi)
fig, ax = plt.subplots(figsize=(10, 6))
gap_means = [panel_results[p]['bayes_minus_floor_gap_mean'] for p in panel_names]
gap_cis = [panel_results[p]['bayes_minus_floor_gap_ci'] for p in panel_names]
gap_ci_low = [ci[0] for ci in gap_cis]
gap_ci_high = [ci[1] for ci in gap_cis]
gap_errs = err_bars(gap_means, gap_cis)

colors = ['crimson' if (lo <= 0 <= hi) else 'seagreen' for lo, hi in zip(gap_ci_low, gap_ci_high)]
ax.barh(panel_names, gap_means, xerr=gap_errs, color=colors, alpha=0.8, capsize=5, edgecolor='black')
ax.axvline(0, color='black', linestyle='--', linewidth=1.5)
ax.set_xlabel('Gap = Bayes-F1 - Floor-F1 (aynı bootstrap CI disiplini)', fontsize=11, fontweight='bold')
ax.set_title('Bayes Tavanı Floor\'u Aşıyor mu? (Kırmızı = CI 0\'ı kapsıyor -> artefakt şüphesi)', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig_path = os.path.join(RESULTS_DIR, 'fig4_gap_vs_floor_artefact_test.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
fig_paths.append(fig_path)
plt.close()

print(f"Tum {len(fig_paths)} sekil kaydedildi: {RESULTS_DIR}")


Tum 4 sekil kaydedildi: /Users/tefe/teknofest_model/teknofest_model/results/v27_bayes_error_robust


In [12]:
# Cell 12: PDF Rapor

class NB45Report(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 15)
        self.cell(0, 10, 'NB45: Saglam Bayes-Error Ceiling (Cok Yontem + Floor-CI Disiplini)', 0, 1, 'C')
        self.set_font('Helvetica', '', 9)
        self.cell(0, 5, f'SEED={SEED} | k-NN(coklu-k) + MST-GHP + Feature-Boot + Floor-CI | {datetime.now().strftime("%Y-%m-%d")}', 0, 1, 'C')
        self.ln(4)

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', '', 8)
        self.cell(0, 10, f'Page {self.page_no()}', 0, 0, 'C')

    def section_title(self, title):
        self.set_font('Helvetica', 'B', 12)
        self.set_fill_color(41, 128, 185)
        self.set_text_color(255, 255, 255)
        self.cell(0, 8, f'  {title}', 0, 1, 'L', fill=True)
        self.set_text_color(0, 0, 0)
        self.ln(2)

    def body_text(self, text):
        self.set_font('Helvetica', '', 9)
        self.multi_cell(0, 5, text)
        self.ln(2)

    def add_table(self, headers, rows, col_widths=None):
        if col_widths is None:
            col_widths = [self.epw / len(headers)] * len(headers)
        self.set_font('Helvetica', 'B', 7)
        self.set_fill_color(52, 73, 94)
        self.set_text_color(255, 255, 255)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, str(h), 1, 0, 'C', fill=True)
        self.ln()
        self.set_font('Helvetica', '', 6.5)
        self.set_text_color(0, 0, 0)
        for j, row in enumerate(rows):
            self.set_fill_color(236, 240, 241) if j % 2 == 0 else self.set_fill_color(255, 255, 255)
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), 1, 0, 'C', fill=True)
            self.ln()
        self.ln(2)

pdf = NB45Report('P', 'mm', 'A4')
pdf.add_page()

pdf.section_title('1. Amac ve Motivasyon')
pdf.body_text(
    "NB40, PAH panelinde tek yontemle (k-NN, tek k-seti) bir Bayes-error tavani hesaplamis ve "
    "gap=+0.1125 bulmustu -- bu, NB21-NB35'in 'PAH plato gercek' bulgusuyla celisiyordu. "
    "Literatur (arXiv 2506.03159), k-NN tabanli Bayes-error tahmininin sinif basi ~100-1000 "
    "ornek gerektirdigini gosteriyor; PAH'in 62 benign ornegi bu esigin altinda. "
    "Bu notebook celiskiyi (a) MST-tabanli GHP ile capraz-dogrulama, (b) feature-altkume "
    "bootstrap ile kirilganlik olcumu, (c) floor-F1'i ayni CI disiplininde hesaplayarak cozer."
)

pdf.section_title('2. Yontemler-Arasi Uzlasma (k-NN vs MST-GHP)')
conv_rows = [[r['Panel'], r['kNN-1 R*_high'], r['GHP Bayes-upper'], r['Disagreement'], r['Reliability (n_min_class)']]
             for r in convergence_rows]
pdf.add_table(['Panel', 'kNN-1 R*_high', 'GHP Bayes-upper', 'Fark', 'Guvenilirlik'], conv_rows,
              col_widths=[30, 38, 38, 30, 54])

pdf.section_title('3. Nihai Ozet: Floor vs Bayes vs Model (Ayni CI Disiplini)')
final_rows = [[r['Panel'], r['Floor-F1 [CI]'], r['Bayes-F1 [CI]'], r['Model-F1'],
               r['Gap(Bayes-Model)'], r['Bayes-Tavan-Guvenilirligi']] for r in summary_rows]
pdf.add_table(['Panel', 'Floor-F1[CI]', 'Bayes-F1[CI]', 'Model-F1', 'Gap(B-M)', 'Guvenilirlik Bayragi'],
              final_rows, col_widths=[18, 42, 42, 22, 22, 44])

pdf.section_title('4. PAH Ozel Karar')
pah_ci_overlap = not (pah_res['bayes_f1_ci'][0] > pah_res['floor_f1_ci'][1] or pah_res['floor_f1_ci'][0] > pah_res['bayes_f1_ci'][1])
pah_verdict = (
    "ARTEFAKT: Bayes-F1 CI, Floor-F1 CI ile cakisiyor (veya guvenilirlik LOW). NB40'in gap=+0.1125 "
    "bulgusu k-NN artefaktidir. PAH plato bulgusu (NB21-NB35, danisman floor-analizi) GECERLIGINI KORUR."
    if (pah_ci_overlap or pah_res['reliability'] == 'LOW') else
    "GERCEK MARJ: Bayes-F1 CI, Floor-F1'den anlamli sekilde ayrisiyor. Reverse-distribution + "
    "TabPFN denenmeden PAH kapatilmamali (yol haritasi #2, #3)."
)
pdf.body_text(
    f"n_benign={pah_res['n_benign']}, reliability={pah_res['reliability']}. "
    f"Floor-F1={pah_res['floor_f1_mean']:.3f} {pah_res['floor_f1_ci']}, "
    f"Bayes-F1={pah_res['bayes_f1_mean']:.3f} {pah_res['bayes_f1_ci']}, "
    f"Model-F1={pah_res['model_f1']:.3f}.\n\nKARAR: {pah_verdict}"
)

pdf.add_page()
pdf.section_title('5. Gorsellestirilmis Sonuclar')
for i, fig_path in enumerate(fig_paths, 1):
    pdf.cell(0, 5, f'Sekil {i}: {os.path.basename(fig_path)}', 0, 1)
    try:
        pdf.image(fig_path, x=10, w=190)
    except Exception:
        pdf.body_text(f'[Sekil {i} yuklenemedi]')
    pdf.ln(1)

report_path = os.path.join(PROJECT_ROOT, 'reports', 'NB45_bayes_error_robust_report.pdf')
pdf.output(report_path)
print(f"PDF Report saved: {report_path}")


PDF Report saved: /Users/tefe/teknofest_model/teknofest_model/reports/NB45_bayes_error_robust_report.pdf
